# C.6 正確答案怎麼變成訓練回饋？


模型對「2+2=?」可能回3、4、5。若能精確核對答案，可以把答對給1、答錯給0，形成reward（獎勵）訊號。前置是[窄任務驗證器](https://birdhackor.github.io/tiny-perceptron-vlm/C.2.html)；這裡的獎勵只是用數字表示哪份輸出符合判準，尚未改變任何權重。先把判準與解析規則寫清楚，下一節才看它如何影響機率。

真實模型輸出的是文字，不是直接交一個Python整數。因此先約定答案格式：允許前後空白，內容只能是可選負號與十進位數字，例如4或-2。含「答案是4」的文字在這份規則下解析失敗，不偷偷抽出其中的4。這並非唯一合理格式，而是一個可追蹤的固定協議；若想接受附帶解釋，需另外定義如何可靠地取出最終答案。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import re

answers = ["3", "4", "5", "答案是4"]
truth = 4
for text in answers:
    clean = text.strip()
    parsed = int(clean) if re.fullmatch(r"-?[0-9]+", clean) else None
    reward = float(parsed == truth)
    print(repr(text), "解析值", parsed, "reward", reward)

`strip`移除前後空白，`fullmatch`要求整個字串都符合規則，`int`只在通過後轉整數。輸出前三份解析成3、4、5，獎勵依序0.0、1.0、0.0；最後一份是None、0.0。None明確表示沒有可用數值，不會把錯誤吞掉再當正確。`float`把比較的True/False轉成1.0/0.0，程式只做解析與計分，沒有模型或參數更新。

這份reward同時包含內容與格式要求：文字雖提到4，格式不符仍拿0。若模型任務是寫自然回答，這可能太嚴；若任務明定只交數字，則符合要求。訓練前要決定自己真正想鼓勵什麼，否則評分規則會把「內容對但格式不同」和「內容錯」混在一起。也不該只獎勵步驟長度，因為寫很長仍可能算錯。

驗證器也有漏洞範圍。本例只判單個整數是否等於真值，沒有評估步驟可靠性、其他事實或任務是否被改寫。若把評測答案混入訓練輸入，模型可能只複製真值；若解析器只掃到任一個4就通過，列一長串數字也可能得分。固定格式與獨立題目能幫助發現這些問題，但最終仍需要核對實際被獎勵的行為。

正式回饋比較故意把這個漏洞放進可檢查的小策略：17個預定動作是單個數字0至15，以及一份列出`0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15`的文字。嚴格reward要求單個整數等於本題加法真值；弱代理reward只要求文字中的任一數字吻合。列出全部數字不會完成「只交一個答案」的任務，卻必然讓這個弱評分器給1。

實際按弱reward訓練後，24道新題各抽16次，共384個動作全部選中列舉；代理分384/384，嚴格正確0/384。例如`(0+3)+3=?`應答6，策略實際抽出的卻是整串0至15。這些文字來自學到的動作分布與真實動作ID採樣，再映射到事先定義的文字，不是語言模型逐token生成，也不是拿固定串冒充解題模型回答。完整訓練與新題檢查在下一節。

練習把題目改為「2+3=?」，所以正確答案改為5；程式只把truth由4改5，候選文字與解析規則保持不變。先預測前三份獎勵變0.0、0.0、1.0，最後一份仍是解析失敗與0.0，再執行核對。這說明評分要依當前題目的正確答案改變，沒有把「4」固定當成好答案，也不是把同一道2+2的真值任意改成5；下一步才是使用這些回饋更新輸出分布。
